# Advanced PEFT Techniques
While standard LoRA and QLoRA dominate production SFT pipelines, specific algorithmic variants address key edge-case limits in convergence rate, rank efficiency, and dynamic multi-tenant serving.

## DoRA

## Weight-Decomposed Low-Rank Adaptation (DoRA)

Standard LoRA updates weights via a single additive delta $\Delta W = B \cdot A$. However, analyzing full fine-tuning (FFN) reveals that full-parameter gradient updates alter both the magnitude ($m$) and the direction ($V$) of weight vectors independently, whereas LoRA exhibits a rigid proportional coupling between magnitude and direction updates.

### Mathematical Formulation

DoRA decomposes the pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$ into a scale magnitude vector $m \in \mathbb{R}^{1 \times k}$ and a directional unit-matrix $V \in \mathbb{R}^{d \times k}$:

$$W = m \odot \frac{V}{\Vert{}V\Vert{}_c}$$

Where:
* $\Vert{}V\Vert{}_c$ represents the column-wise $\mathcal{L}_2$ norm of $V$
* $\odot$ denotes column-wise vector scaling

### Injecting Low-Rank Updates into Direction $V$

Instead of updating $W_0$ directly, DoRA freezes the directional baseline $V = W_0$ and applies a LoRA update specifically to the directional component:

$$W_{\text{DoRA}} = m \odot \frac{W_0 + \frac{\alpha}{r}(B \cdot A)}{\Vert{}W_0 + \frac{\alpha}{r}(B \cdot A)\Vert{}_c}$$

Where:
* **$m \in \mathbb{R}^{1 \times k}$:** Trainable magnitude parameter initialized to $m = \Vert{}W_0\Vert{}_c$
* **$W_0$:** Frozen base weight matrix
* **$B, A$:** Standard trainable LoRA matrices initialized to zero and Gaussian distribution respectively

### Magnitude & Direction in LLMs vs. Physics

**In Physics:**

* **Magnitude:** The speed of a car ($60\text{ mph}$)
* **Direction:** The compass heading ($\text{North-East}$)

**In a Linear Layer of an LLM** ($W \in \mathbb{R}^{d \times k}$):

Think of the weight matrix $W$ as a collection of $k$ column vectors. Each column $w_j \in \mathbb{R}^d$ connects all $d$ input features to Output Neuron $j$.

```
                     Weight Matrix W (d x k)
                     ┌─────┬─────┬───┬─────┐
                  d  │ w_1 │ w_2 │...│ w_k │  <- Each column is a 
                 rows│     │     │   │     │     vector for 1 neuron
                     └─────┴─────┴───┴─────┘
                       Col 1 Col 2     Col k
```

When we decompose a single column vector $w_j$ into Magnitude and Direction:

$$\text{Magnitude } (m_j) = \Vert{}w_j\Vert{}_2 = \sqrt{w_{1,j}^2 + w_{2,j}^2 + \dots + w_{d,j}^2}$$

$$\text{Direction } (v_j) = \frac{w_j}{\Vert{}w_j\Vert{}_2} \quad (\text{A unit vector with length } = 1.0)$$

### Functional Meaning in LLMs

**Direction** ($v_j$): **WHAT pattern the neuron listens for**

* The direction vector defines the orientation in $d$-dimensional embedding space
* It dictates which specific combination of input tokens or features will trigger this neuron (e.g., listening for "legal terminology" vs. "Python syntax")

**Magnitude** ($m_j$): **HOW STRONGLY the neuron fires** (Feature Gain/Scaling)

* Magnitude controls the scale or amplification of that neuron's output activation
* A high magnitude means "if you detect this pattern, amplify it strongly into the next layer"

---

### The Core Problem with Standard LoRA

When NVIDIA researched full fine-tuning (FT) vs. PEFT (LoRA) for the DoRA paper (ICML 2024), they analyzed how weight matrices evolve:

**Full Fine-Tuning (FT):**
* Models frequently make subtle directional rotations while keeping magnitude small, OR significantly change feature magnitude without drastically rotating direction
* Direction and magnitude updates are decoupled

**Standard LoRA** ($W' = W_0 + B \cdot A$):
* Adding a delta matrix $\Delta W = B \cdot A$ directly onto $W_0$ couples magnitude and direction together
* Whenever LoRA rotates the direction vector, the addition of $B \cdot A$ proportionally inflates or stretches the magnitude

---

### Concrete Numerical Example: LoRA vs. DoRA

Imagine a simple linear layer with $d=2$ inputs connecting to a single output neuron.

**Pre-Trained Base Weight Vector:**

$$w_0 = \begin{bmatrix} 3.0 \\ 4.0 \end{bmatrix}$$

* **Base Magnitude** ($m_0$): $\sqrt{3^2 + 4^2} = \mathbf{5.0}$
* **Base Direction** ($v_0$): $\frac{1}{5.0} \begin{bmatrix} 3.0 \\ 4.0 \end{bmatrix} = \mathbf{\begin{bmatrix} 0.6 \\ 0.8 \end{bmatrix}}$ (Unit vector, length = $1.0$)

**Scenario:** We want to ROTATE the neuron's detector direction slightly, but KEEP its firing magnitude at $5.0$.

---

**Attempt A: Standard LoRA**

LoRA tries to change the direction by adding an adapter update $\Delta W = B \cdot A = \begin{bmatrix} 1.0 \\ 0.0 \end{bmatrix}$.

$$w_{\text{LoRA}} = w_0 + \Delta W = \begin{bmatrix} 3.0 \\ 4.0 \end{bmatrix} + \begin{bmatrix} 1.0 \\ 0.0 \end{bmatrix} = \begin{bmatrix} 4.0 \\ 4.0 \end{bmatrix}$$

Let's check what happened to magnitude and direction:

* **New Direction:** $\frac{1}{\sqrt{32}} \begin{bmatrix} 4.0 \\ 4.0 \end{bmatrix} \approx \mathbf{\begin{bmatrix} 0.707 \\ 0.707 \end{bmatrix}}$ (Success: Direction rotated from $[0.6, 0.8]$ to $[0.707, 0.707]$)
* **New Magnitude:** $\sqrt{4^2 + 4^2} = \sqrt{32} \approx \mathbf{5.65}$ (UNINTENDED SIDE EFFECT: Magnitude inflated from $5.0 \to 5.65$!)

**Problem:** Because LoRA uses simple addition ($W_0 + BA$), you cannot rotate direction without unintentionally changing magnitude.

---

**Attempt B: DoRA (Weight-Decomposed Low-Rank Adaptation)**

DoRA explicitly splits the formula into two independent trainable terms:

$$w_{\text{DoRA}} = m \odot \frac{w_0 + B \cdot A}{\Vert{}w_0 + B \cdot A\Vert{}_2}$$

Where:
* $m$ is a standalone scalar magnitude parameter (initialized to $m = 5.0$)
* The directional vector is explicitly normalized to length $1.0$ inside the equation before scaling by $m$!

Now, applying the same adapter update $\Delta W = B \cdot A = \begin{bmatrix} 1.0 \\ 0.0 \end{bmatrix}$:

**Calculate Updated Unnormalized Direction:**

$$v_{\text{temp}} = \begin{bmatrix} 3.0 \\ 4.0 \end{bmatrix} + \begin{bmatrix} 1.0 \\ 0.0 \end{bmatrix} = \begin{bmatrix} 4.0 \\ 4.0 \end{bmatrix}$$

**Normalize Direction back to Unit Length** $1.0$:

$$v_{\text{norm}} = \frac{1}{\sqrt{32}} \begin{bmatrix} 4.0 \\ 4.0 \end{bmatrix} = \begin{bmatrix} 0.707 \\ 0.707 \end{bmatrix}$$

**Scale by Independent Magnitude** $m$ ($m=5.0$):

$$w_{\text{DoRA}} = 5.0 \times \begin{bmatrix} 0.707 \\ 0.707 \end{bmatrix} = \begin{bmatrix} 3.535 \\ 3.535 \end{bmatrix}$$

**Check DoRA's Results:**

* **New Direction:** $\begin{bmatrix} 0.707 \\ 0.707 \end{bmatrix}$ (Direction rotated perfectly)
* **New Magnitude:** $\sqrt{3.535^2 + 3.535^2} = \mathbf{5.0}$ (Magnitude remained EXACTLY $5.0$!)

---

### Key Takeaway for Implementation

* **DoRA adds a tiny** $1\text{D}$ vector parameter $m$ ($1 \times k$, one magnitude scalar per output column) and divides direction by its column norm during the forward pass
* **Why it matters:** This allows DoRA to mimic full fine-tuning capacity at low ranks ($r=4$ or $r=8$) because the optimizer can tweak how strongly a feature fires ($m$) independently from what pattern the feature detects ($B \cdot A$)

## AdaLoRA: Dynamic Rank Allocation Across Layers

**Standard LoRA Problem:**

Standard LoRA assigns a fixed rank $r$ (e.g., $r=16$) uniformly to all selected projection matrices across all layers.

**Problem Statement:**

Not all layers or projections contribute equally to task adaptation. Attention projections in middle layers may require higher rank capacity, while early MLP projections need minimal adaptation. Fixed allocation wastes parameter budget.

---

### AdaLoRA Mechanics

AdaLoRA parameterizes the low-rank update using Singular Value Decomposition (SVD):

$$\Delta W = P \cdot \Lambda \cdot Q$$

Where:
* $P \in \mathbb{R}^{d \times r}$ and $Q \in \mathbb{R}^{r \times k}$ are orthogonal matrices
* $\Lambda \in \mathbb{R}^{r \times r}$ is a diagonal matrix containing singular values $\lambda_i$

### Dynamic Training Process

**During training:**

1. The optimizer updates $P, \Lambda, Q$
2. An importance score $S_i$ is tracked for each singular value $\lambda_i$ based on magnitude and gradient magnitude
3. At regular intervals, low-importance singular values $\lambda_i$ are pruned (zeroed out)
4. The parameter budget dynamically redistributes rank $r_l$ to high-importance layers, maximizing overall model capacity for a fixed parameter budget